# Preparación de Datos y Feature Engineering

**Integrantes:** Juan Camilo Conrado | Sergio Cadavid | Mateo Chang  
**Curso:** Machine Learning para Ciencia de Datos — Segundo Entregable

## 1. Cargar Librerías

In [28]:
!pip install xgboost statsmodels imbalanced-learn optuna deap lime

!pip install xgboost statsmodels imbalanced-learn optuna deap lime seaborn

In [29]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import warnings
warnings.filterwarnings('ignore')

# Sklearn - Preprocesamiento y Pipelines
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Sklearn - Modelos de Regresión
from sklearn.linear_model import Ridge, Lasso, LogisticRegression
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.svm import SVR, SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

# Sklearn - Validación y Métricas
from sklearn.model_selection import (TimeSeriesSplit, GridSearchCV,
                                     RandomizedSearchCV)
from sklearn.metrics import (mean_squared_error, mean_absolute_error, r2_score,
                             accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, roc_curve,
                             confusion_matrix, classification_report)

# XGBoost
from xgboost import XGBRegressor, XGBClassifier

# Statsmodels - Tests estadísticos
from statsmodels.stats.diagnostic import het_white, acorr_ljungbox
from statsmodels.tsa.stattools import bds, acf
from statsmodels.stats.stattools import jarque_bera
import statsmodels.api as sm

# Balanceo de clases
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.pipeline import Pipeline as ImbPipeline

# Optimización Bayesiana
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Algoritmos Genéticos
from deap import base, creator, tools, algorithms

# LIME
import lime
import lime.lime_tabular

# Scipy
from scipy import stats

print("Todas las librerías cargadas correctamente.")

Todas las librerías cargadas correctamente.


## 2. Cargar y Limpiar Datos

In [30]:
df = pd.read_csv("../src/intc.us.txt")
df.columns = [c.strip().lower() for c in df.columns]
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.sort_values("date").reset_index(drop=True)
df = df.dropna(subset=["date", "open", "high", "low", "close", "volume"]).copy()

print("Duplicados en date:", df["date"].duplicated().sum())

if "openint" in df.columns:
    df = df.drop(columns=["openint"])

print(df.shape)
df.head()

Duplicados en date: 0
(11556, 6)


,date,open,high,low,close,volume
0,1972-01-07,0.01592,0.01592,0.01592,0.01592,3787746
1,1972-01-14,0.00791,0.00791,0.00791,0.00791,7878523
2,1972-01-21,0.00791,0.00791,0.00791,0.00791,1060564
3,1972-01-24,0.00791,0.00791,0.00791,0.00791,6060405
4,1972-01-25,0.00791,0.00791,0.00791,0.00791,1060564


---
> **Interpretación:** La carga confirma **11,556 observaciones sin duplicados ni nulos** en las columnas esenciales. Los precios históricos muy bajos (Open/Close ~0.016 en los 70s) reflejan ajustes por splits accionarios. La estructura OHLCV está íntegra y lista para procesamiento.

## 3. Filtrar Período Moderno
Se filtran datos desde 1990 para evitar tramos históricos con cierres repetidos que distorsionan el cálculo de retornos y volatilidad.

In [31]:
df = df[df["date"] >= "1990-01-01"].copy()
df = df.sort_values("date").reset_index(drop=True)
print(f"Período: {df['date'].min().date()} -> {df['date'].max().date()}")
print(f"Observaciones: {len(df)}")

Período: 1990-01-02 -> 2017-11-10
Observaciones: 7022


---
> **Interpretación:** Al filtrar desde 1990, el dataset queda con **7,022 observaciones** (27 años). Se elimina el período 1972–1989 porque Intel tenía precios extremadamente bajos y repetidos (stock en etapas tempranas), lo que distorsionaría el cálculo de retornos logarítmicos y volatilidades. El período moderno captura mercados más maduros y líquidos.

## 4. Retornos Logarítmicos y Volatilidad Rolling

In [32]:
df["log_ret"] = np.log(df["close"]).diff()

df["vol_7"] = df["log_ret"].rolling(7).std()
df["vol_14"] = df["log_ret"].rolling(14).std()
df["vol_21"] = df["log_ret"].rolling(21).std()
df["vol_28"] = df["log_ret"].rolling(28).std()

print(df[["date", "log_ret", "vol_7", "vol_14", "vol_21", "vol_28"]].tail(10))

           date   log_ret     vol_7    vol_14    vol_21    vol_28
7012 2017-10-30 -0.000680  0.025761  0.019486  0.016644  0.014514
7013 2017-10-31  0.024941  0.025870  0.019325  0.016605  0.014914
7014 2017-11-01  0.026468  0.025883  0.019365  0.017127  0.015265
7015 2017-11-02  0.008321  0.025393  0.019398  0.016999  0.015170
7016 2017-11-03 -0.016277  0.027629  0.020720  0.017834  0.015874
7017 2017-11-06  0.013582  0.027637  0.020518  0.017841  0.015872
7018 2017-11-07  0.001712  0.015072  0.020697  0.017890  0.015917
7019 2017-11-08 -0.001712  0.015179  0.020578  0.017775  0.016017
7020 2017-11-09 -0.008602  0.014261  0.021255  0.017762  0.015886
7021 2017-11-10 -0.015673  0.011507  0.022344  0.018368  0.016412


---
> **Interpretación:** Se calculan retornos logarítmicos diarios y volatilidades rolling en 4 ventanas (7, 14, 21, 28 días). Al final del período (octubre–noviembre 2017), INTC muestra una volatilidad de 7 días de ~2.6% y de 21 días de ~1.7% — niveles relativamente bajos, indicando un período de calma antes del cierre del dataset. La volatilidad de ventana mayor es siempre menor (efecto de suavizado).

## 5. Definición de Targets — Corrección del Shift

**Problema anterior:** Se usaba `vol_7.shift(-1)`, generando overlap de 6/7 días entre ventana actual y futura (R² inflado a ~0.71).

**Corrección:** `shift(-7)` para predecir volatilidad futura **sin overlap**. Análogamente shift(-14) y shift(-21).

**Target de clasificación:** Variable binaria que indica si la volatilidad futura será **mayor** que la actual (régimen de volatilidad creciente).

In [33]:
# --- Targets de REGRESIÓN (sin overlap) ---
df["target_vol_7"] = df["vol_7"].shift(-7)
df["target_vol_14"] = df["vol_14"].shift(-14)
df["target_vol_21"] = df["vol_21"].shift(-21)

# --- Target de CLASIFICACIÓN ---
# 1 = la volatilidad futura sube respecto a la actual, 0 = baja o se mantiene
df["target_class"] = (df["target_vol_7"] > df["vol_7"]).astype(int)

print("Distribución del target de clasificación:")
print(df["target_class"].value_counts())
print(f"\nProporción clase 1: {df['target_class'].mean():.3f}")

Distribución del target de clasificación:
target_class
0    3608
1    3414
Name: count, dtype: int64

Proporción clase 1: 0.486


---
> **Interpretación:** El **target de clasificación** tiene 3,608 instancias de clase 0 (volatilidad baja/igual) vs 3,414 de clase 1 (volatilidad sube), con proporción de clase 1 del **48.6%**. Esto indica un dataset **casi perfectamente balanceado**, lo que simplifica el problema de clasificación y hace que las técnicas de SMOTE/ADASYN tengan impacto mínimo. Para regresión, los 3 targets son: volatilidad futura a 7, 14 y 21 días sin overlap (shift correcto).

## 6. Feature Engineering — 52 Variables

Se construyen features que capturan estructura temporal a múltiples escalas:
- Lags de retornos logarítmicos y de volatilidades (4 ventanas)
- Retorno absoluto promedio (proxy de volatilidad realizada)
- Rango High-Low rolling, retorno acumulado
- Ratios de volatilidad (estructura temporal)
- Variables OHLCV derivadas

In [34]:
# Lags de retorno logarítmico
for lag in [1, 2, 3, 5, 7, 10, 14, 21]:
    df[f"log_ret_lag{lag}"] = df["log_ret"].shift(lag)

# Lags de volatilidades (todas las ventanas)
for v in [7, 14, 21, 28]:
    for lag in [1, 2, 3, 5, 7, 14, 21]:
        df[f"vol_{v}_lag{lag}"] = df[f"vol_{v}"].shift(lag)

# Retorno absoluto promedio
for w in [5, 10, 20]:
    df[f"abs_ret_ma{w}"] = df["log_ret"].abs().rolling(w).mean()

# Variables OHLCV derivadas
df["hl_range"] = (df["high"] - df["low"]) / df["close"]
df["oc_change"] = (df["close"] - df["open"]) / df["open"]
df["vol_change"] = df["volume"].pct_change()
df["vol_ma_10"] = df["volume"].rolling(10).mean()
df["vol_ratio_10"] = df["volume"] / df["vol_ma_10"]

# Range rolling
for w in [5, 10, 20]:
    df[f"hl_range_ma{w}"] = df["hl_range"].rolling(w).mean()

# Retorno acumulado
for w in [5, 10, 20]:
    df[f"cum_ret_{w}"] = df["log_ret"].rolling(w).sum()

# Ratios de volatilidad (estructura temporal)
df["vol_ratio_7_14"] = df["vol_7"] / df["vol_14"]
df["vol_ratio_7_21"] = df["vol_7"] / df["vol_21"]
df["vol_ratio_14_28"] = df["vol_14"] / df["vol_28"]

# Reemplazar infinitos
df = df.replace([np.inf, -np.inf], np.nan)

# Definir feature columns
exclude_cols = ["date", "open", "high", "low", "close", "volume",
                "log_ret", "vol_7", "vol_14", "vol_21", "vol_28",
                "target_vol_7", "target_vol_14", "target_vol_21",
                "target_class", "vol_ma_10"]
feature_cols = [c for c in df.columns if c not in exclude_cols]

print(f"Total features: {len(feature_cols)}")

Total features: 52


---
> **Interpretación:** Se construyen **52 variables** agrupadas en 6 categorías: lags de retornos (8 variables), lags de volatilidades en 4 ventanas × 7 lags (28 variables), promedios de retorno absoluto (3), derivadas OHLCV (5), rangos rolling (3) y retornos acumulados (3), más ratios de volatilidad (3). Esta riqueza de features permite al modelo capturar estructura temporal a múltiples escalas — desde el día anterior hasta 21 días atrás.

## 7. Dataset Modelable y Split Temporal 75/25

In [35]:
# Dataset para regresión (target vol_7)
df_reg = df.dropna(subset=feature_cols + ["target_vol_7"]).copy()
df_reg = df_reg.sort_values("date").reset_index(drop=True)

# Dataset para clasificación
df_clf = df.dropna(subset=feature_cols + ["target_class"]).copy()
df_clf = df_clf.sort_values("date").reset_index(drop=True)

# --- Split regresión ---
cut_r = int(len(df_reg) * 0.75)
train_r = df_reg.iloc[:cut_r].copy()
test_r = df_reg.iloc[cut_r:].copy()

X_train_r = train_r[feature_cols]
X_test_r = test_r[feature_cols]
y_train_r = train_r["target_vol_7"]
y_test_r = test_r["target_vol_7"]

# --- Split clasificación ---
cut_c = int(len(df_clf) * 0.75)
train_c = df_clf.iloc[:cut_c].copy()
test_c = df_clf.iloc[cut_c:].copy()

X_train_c = train_c[feature_cols]
X_test_c = test_c[feature_cols]
y_train_c = train_c["target_class"]
y_test_c = test_c["target_class"]

print(f"=== REGRESIÓN ===")
print(f"Train: {train_r['date'].min().date()} -> {train_r['date'].max().date()} | n={len(train_r)}")
print(f"Test:  {test_r['date'].min().date()} -> {test_r['date'].max().date()} | n={len(test_r)}")

print(f"\n=== CLASIFICACIÓN ===")
print(f"Train: {train_c['date'].min().date()} -> {train_c['date'].max().date()} | n={len(train_c)}")
print(f"Test:  {test_c['date'].min().date()} -> {test_c['date'].max().date()} | n={len(test_c)}")
print(f"Distribución train: {dict(y_train_c.value_counts())}")
print(f"Distribución test:  {dict(y_test_c.value_counts())}")

=== REGRESIÓN ===
Train: 1990-03-13 -> 2010-12-01 | n=5223
Test:  2010-12-02 -> 2017-11-01 | n=1742

=== CLASIFICACIÓN ===
Train: 1990-03-13 -> 2010-12-09 | n=5229
Test:  2010-12-10 -> 2017-11-10 | n=1743
Distribución train: {0: np.int64(2673), 1: np.int64(2556)}
Distribución test:  {0: np.int64(902), 1: np.int64(841)}


---
> **Interpretación:** El split temporal 75/25 produce:
> - **Regresión:** Train (5,223 días, 1990–2010) / Test (1,742 días, 2010–2017)
> - **Clasificación:** Train (5,229 días) / Test (1,743 días)
>
> La distribución de clases en train (2,673 vs 2,556) y test (902 vs ...) es casi uniforme en ambos subconjuntos, confirmando el balance natural. El período de test 2010–2017 incluye años de recuperación post-crisis y el auge tecnológico — un escenario desafiante y representativo.